In [1]:
import asyncio
import json
import logging
import copy
from typing import Any, Dict, List, Optional
from dataclasses import dataclass
import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
import time
import nest_asyncio
from concurrent.futures import ThreadPoolExecutor

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging for notebook
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful!")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
print(f"🔥 Number of GPUs: {torch.cuda.device_count()}")

INFO 07-10 05:39:59 [__init__.py:244] Automatically detected platform cuda.
✅ Imports successful!
🔥 CUDA available: True
🔥 Number of GPUs: 8


In [2]:
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')
for _ in sys.path:
    print(_)

from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator
print("✅ Conversation Generator class defined")

/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/ray/thirdparty_files
/home/sagemaker-user/.conda/envs/collabllm/lib/python310.zip
/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10
/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/lib-dynload

/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages
/tmp/tmpiurbl6q7
/home/sagemaker-user/csbai/multiturn_rl
✅ Conversation Generator class defined


In [3]:
test_config = ConversationConfig(
    assistant_meta_prompt="You are a helpful cooking assistant. Provide clear, step-by-step cooking instructions and tips.",
    user_meta_prompt="You are a user asking an assistant about the following things. Generate response based on the following conversation: \n {chat_history}",
    max_total_turns=8,
    max_gen_workers=2,
    local_model_path="/home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test",  # Your LoRA adapter path
    base_model_path="meta-llama/Llama-3.2-1B-Instruct",  # Your base model path
    assistant_generation_kwargs={
        "temperature": 0.7,
        "max_tokens": 256
    },
    user_generation_kwargs={
        "model": "anthropic.claude-3-sonnet-20240229-v1:0",  # or your preferred bedrock model
        "temperature": 1.0,
        "max_tokens": 512,
        "num_retries": 10  # For UserSimulator's retry logic
    },
    batch_size=5,
    enable_batching=True
)

print("✅ Test configuration created")
print(f"📝 Task: {test_config.user_meta_prompt}")
print(f"🔄 Max turns: {test_config.max_total_turns}")
print(f"⚡ Max workers: {test_config.max_gen_workers}")
print(f"📦 Batch size: {test_config.batch_size}")  # CHANGED: Added batch info

✅ Test configuration created
📝 Task: You are a user asking an assistant about the following things. Generate response based on the following conversation: 
 {chat_history}
🔄 Max turns: 8
⚡ Max workers: 2
📦 Batch size: 5


In [6]:
async def test_single_conversation():
    """Test generating a single conversation"""
    print("🧪 Testing single conversation generation...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    test_prompt = "How do I make chocolate chip cookies?"
    
    conversation = await generator.generate_single_conversation(test_prompt)
    
    if conversation:
        print("\n" + "="*50)
        print("📋 GENERATED CONVERSATION:")
        print("="*50)
        
        for i, msg in enumerate(conversation):
            role_emoji = {"system": "⚙️", "user": "👤", "assistant": "🤖"}
            print(f"\n{role_emoji.get(msg['role'], '❓')} {msg['role'].upper()}:")
            print(f"   {msg['content']}")
        
        print("\n" + "="*50)
        print(f"✅ Success! Generated {len(conversation)} messages")
        return conversation
    else:
        print("❌ Failed to generate conversation")
        return None

# Run the test
print("🚀 Starting single conversation test...")
single_conv_result = await test_single_conversation()

🚀 Starting single conversation test...
🧪 Testing single conversation generation...
🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-10 03:06:19 [config.py:823] This model supports multiple tasks: {'classify', 'embed', 'generate', 'score', 'reward'}. Defaulting to 'generate'.
INFO 07-10 03:06:19 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-10 03:06:19 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.


INFO 07-10 03:06:19 [core.py:455] Waiting for init message from front-end.
INFO 07-10 03:06:19 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, served_model_name=meta-llama/Llama-3.2-1B

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 07-10 03:06:20 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f83cc35ff70>
(VllmWorker rank=0 pid=30347) INFO 07-10 03:06:20 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_11f5457b'), local_subscribe_addr='ipc:///tmp/4fb030e9-2ace-4ec7-858e-3248f5f93a59', remote_subscribe_addr=None, remote_addr_ipv6=False)
WARNING 07-10 03:06:20 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f83cc35fc10>
(VllmWorker rank=1 pid=30351) INFO 07-10 03:06:20 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_c07ee23d'), local_subscribe_addr='ipc:///tmp/0860edbb-4622-48e7-a334-8

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Turn 3: Generating user response...
++++++++++++++++++++++ User input prompt: ++++++++++++++++++++++
 [{'role': 'user', 'content': "You are a user asking an assistant about the following things. Generate response based on the following conversation: \n User: How do I make chocolate chip cookies?\nAssistant: 1. Preheat your oven to 375°F (190°C). Line a baking sheet with parchment paper or a silicone mat. \n2. In a large bowl, whisk together 2 1/4 cups of all-purpose flour, 1 tsp baking soda, 1 tsp salt, and 1 cup of granulated sugar. \n3. In another bowl, use an electric mixer to cream 1/2 cup of unsalted butter until it's light and fluffy. \n4. Add 2 large eggs to the butter mixture and mix until combined. \n5. Stir in 1 cup of brown sugar and 1 cup of semi-sweet chocolate chips. \n6. Gradually add the dry ingredients to the wet ingredients and mix until a dough forms. \n7. Scoop tablespoon-sized balls of dough onto the prepared baking sheet, leaving about 2 inches of space between 

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Turn 4: Generating user response...
++++++++++++++++++++++ User input prompt: ++++++++++++++++++++++
 [{'role': 'user', 'content': "You are a user asking an assistant about the following things. Generate response based on the following conversation: \n User: How do I make chocolate chip cookies?\nAssistant: 1. Preheat your oven to 375°F (190°C). Line a baking sheet with parchment paper or a silicone mat. \n2. In a large bowl, whisk together 2 1/4 cups of all-purpose flour, 1 tsp baking soda, 1 tsp salt, and 1 cup of granulated sugar. \n3. In another bowl, use an electric mixer to cream 1/2 cup of unsalted butter until it's light and fluffy. \n4. Add 2 large eggs to the butter mixture and mix until combined. \n5. Stir in 1 cup of brown sugar and 1 cup of semi-sweet chocolate chips. \n6. Gradually add the dry ingredients to the wet ingredients and mix until a dough forms. \n7. Scoop tablespoon-sized balls of dough onto the prepared baking sheet, leaving about 2 inches of space between 

In [7]:
single_conv_result

[{'role': 'user', 'content': 'How do I make chocolate chip cookies?'},
 {'role': 'assistant',
  'content': "1. Preheat your oven to 375°F (190°C). Line a baking sheet with parchment paper or a silicone mat. \n2. In a large bowl, whisk together 2 1/4 cups of all-purpose flour, 1 tsp baking soda, 1 tsp salt, and 1 cup of granulated sugar. \n3. In another bowl, use an electric mixer to cream 1/2 cup of unsalted butter until it's light and fluffy. \n4. Add 2 large eggs to the butter mixture and mix until combined. \n5. Stir in 1 cup of brown sugar and 1 cup of semi-sweet chocolate chips. \n6. Gradually add the dry ingredients to the wet ingredients and mix until a dough forms. \n7. Scoop tablespoon-sized balls of dough onto the prepared baking sheet, leaving about 2 inches of space between each cookie. \n8. Bake for 10-12 minutes or until the edges are lightly golden brown. \n9. Remove the cookies from the oven and let them cool on the baking sheet for 5 minutes before transferring them to

In [4]:
async def test_batch_conversations():
    """Test generating multiple conversations in parallel - ENHANCED"""
    print("🧪 Testing batch conversation generation...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    # CHANGED: Start with fewer prompts for debugging
    test_prompts = [
        "How do I make chocolate chip cookies?",
        "What's the best way to cook pasta?",
        "How can I make a simple salad?"
    ] * 20
    
    print(f"📝 Testing with {len(test_prompts)} prompts...")
    
    # CHANGED: Use new batch generation method with debugging
    start_time = time.time()
    
    print("🔍 Starting batch generation...")
    results = await generator.generate_conversations_batch(test_prompts, len(test_prompts))
    
    end_time = time.time()
    
    # Filter out None results
    successful_conversations = [conv for conv in results if conv is not None]
    
    print(f"\n⏱️  Total time: {end_time - start_time:.2f} seconds")
    print(f"✅ Successfully generated {len(successful_conversations)}/{len(test_prompts)} conversations")
    
    # Show summary
    for i, conv in enumerate(successful_conversations):
        if conv:
            print(f"   Conversation {i+1}: {len(conv)} messages")
            if i < len(test_prompts):
                print(f"      Prompt: {test_prompts[i][:50]}...")
    
    return successful_conversations

# ADDED: Simple sequential test for comparison
async def test_sequential_conversations():
    """Test generating conversations one by one for debugging"""
    print("🧪 Testing SEQUENTIAL conversation generation for debugging...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    test_prompts = [
        "How do I make chocolate chip cookies?",
        "What's the best way to cook pasta?"
    ]
    
    results = []
    for i, prompt in enumerate(test_prompts):
        print(f"\n🔄 Starting conversation {i+1}/{len(test_prompts)}")
        start_time = time.time()
        
        result = await generator.generate_single_conversation(prompt)
        
        end_time = time.time()
        print(f"⏱️  Conversation {i+1} took {end_time - start_time:.2f} seconds")
        
        if result:
            print(f"✅ Conversation {i+1} completed with {len(result)} messages")
            results.append(result)
        else:
            print(f"❌ Conversation {i+1} failed")
            results.append(None)
    
    return results

# ADDED: Simple batch test without the complex batching logic
async def test_simple_batch():
    """Test simple concurrent execution without complex batching"""
    print("🧪 Testing SIMPLE batch conversation generation...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    test_prompts = [
        "How do I make chocolate chip cookies?",
        "What's the best way to cook pasta?",
        "How can I make a simple salad?"
    ] 
    
    print(f"📝 Testing with {len(test_prompts)} prompts...")
    
    # SIMPLE: Just create tasks and wait for them
    start_time = time.time()
    
    # Create all tasks
    tasks = []
    for i, prompt in enumerate(test_prompts):
        print(f"📝 Creating task {i+1}: {prompt[:30]}...")
        task = asyncio.create_task(generator.generate_single_conversation(prompt))
        tasks.append(task)
    
    print(f"⏳ Waiting for {len(tasks)} tasks...")
    
    # Wait for all tasks with timeout
    try:
        results = await asyncio.wait_for(asyncio.gather(*tasks), timeout=300)  # 5 minute timeout
        print("✅ All tasks completed!")
    except asyncio.TimeoutError:
        print("❌ Tasks timed out after 5 minutes")
        return []
    
    end_time = time.time()
    
    # Filter successful results
    successful = [r for r in results if r is not None]
    
    print(f"\n⏱️  Total time: {end_time - start_time:.2f} seconds")
    print(f"✅ Successfully generated {len(successful)}/{len(test_prompts)} conversations")
    
    return successful

# # Run the simple batch test first
# print("🚀 Starting SIMPLE batch test...")
# simple_batch_results = await test_simple_batch()

In [5]:
print("\n" + "="*50)
print("🚀 Now testing the complex batch method...")
batch_results = await test_batch_conversations()


🚀 Now testing the complex batch method...
🧪 Testing batch conversation generation...
🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-10 05:40:25 [config.py:823] This model supports multiple tasks: {'classify', 'generate', 'embed', 'score', 'reward'}. Defaulting to 'generate'.
INFO 07-10 05:40:25 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-10 05:40:25 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 07-10 05:40:26 [core.py:455] Waiting for init message from front-end.
INFO 07-10 05:40:26 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, t

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=1 pid=45606) INFO 07-10 05:40:31 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=4 pid=45609) INFO 07-10 05:40:31 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=5 pid=45610) INFO 07-10 05:40:31 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=7 pid=45612) INFO 07-10 05:40:31 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=0 pid=45605) INFO 07-10 05:40:31 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=0 pid=45605) INFO 07-10 05:40:31 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=6 pid=45611) INFO 07-10 05:40:31 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=1 pid=45606) INFO 07-10 05:40:31 [default_loader.py:272] Loading weights took 0.10 seconds
(VllmWorker rank=7 pid=45612) INFO 07-10 05:40:31 [weight_utils.py:345] No model.safetensors.

INFO: ✅ Successfully initialized vLLM with LoRA: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test


✅ Conversation generator ready!
📝 Testing with 60 prompts...
🔍 Starting batch generation...
🚀 Starting batch generation for 60 conversations...
🔄 Round 1: Processing 60 active conversations


...

Adding requests:   0%|          | 0/59 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/59 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  🛑 Conversation 30 terminated by user
🔄 Round 4: Processing 58 active conversations


Adding requests:   0%|          | 0/58 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/58 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
# 🔴 NEW TEST: Test early termination handling
async def test_early_termination():
    """Test conversations with different termination points"""
    print("🧪 Testing early termination handling...")
    
    # Create a custom user simulator that terminates at different points
    class VariableTerminationUserSimulator:
        def __init__(self, termination_turn, task_desc='', single_turn_prompt='', **kwargs):
            self.termination_turn = termination_turn
            self.call_count = 0
            
        def __call__(self, messages: List[dict]) -> str:
            self.call_count += 1
            
            if self.call_count >= self.termination_turn:
                return "Perfect! Thank you for your help!"
            else:
                return f"That's helpful. Can you tell me more about step {self.call_count + 1}?"
    
    # Temporarily modify the generator to use our custom simulator
    generator = MultiTurnConversationGenerator(test_config)
    
    # Create test scenarios with different termination points
    test_scenarios = [
        ("Quick pasta recipe?", 2),  # Terminates after 2 turns
        ("How to make a complex French dish?", 5),  # Terminates after 5 turns
        ("Simple sandwich instructions?", 1),  # Terminates after 1 turn
        ("Detailed cake baking process?", 4),  # Terminates after 4 turns
        ("Quick salad?", 3),  # Terminates after 3 turns
    ]
    
    # Create custom conversation states
    conversation_states = []
    for i, (prompt, term_turn) in enumerate(test_scenarios):
        state = {
            'id': i,
            'prompt': prompt,
            'chat_history': [
                {"role": "system", "content": test_config.task_desc},
                {"role": "user", "content": prompt}
            ],
            'user_sim': VariableTerminationUserSimulator(
                termination_turn=term_turn,
                task_desc=test_config.task_desc,
                single_turn_prompt=prompt
            ),
            'completed': False,
            'turn_count': 0
        }
        conversation_states.append(state)
    
    print(f"📝 Testing {len(test_scenarios)} conversations with different termination points...")
    print("Expected terminations:", [t[1] for t in test_scenarios])
    
    start_time = time.time()
    
    # Process conversations in rounds
    max_rounds = test_config.max_total_turns // 2
    
    for round_idx in range(max_rounds):
        # Filter active conversations
        active_states = [s for s in conversation_states if not s['completed']]
        
        if not active_states:
            print(f"✅ All conversations completed by round {round_idx}")
            break
        
        print(f"🔄 Round {round_idx + 1}: Processing {len(active_states)} active conversations")
        
        # Generate assistant responses in batch
        await generator._process_assistant_turn_batch(active_states)
        
        # Check for terminations and generate user responses
        await generator._process_user_turn_batch(active_states)
        
        # Update turn counts
        for state in active_states:
            state['turn_count'] += 1
    
    end_time = time.time()
    
    # Show results
    print(f"\n⏱️  Total time: {end_time - start_time:.2f} seconds")
    print("\n📊 Results:")
    for i, (state, (prompt, expected_term)) in enumerate(zip(conversation_states, test_scenarios)):
        actual_turns = state['turn_count']
        print(f"   Conv {i+1}: Expected {expected_term} turns, got {actual_turns} turns - {prompt[:30]}...")
        print(f"            Messages in conversation: {len(state['chat_history'])}")
    
    return [state['chat_history'] for state in conversation_states]

# Run early termination test
early_term_results = await test_early_termination()

🧪 Testing early termination handling...
🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-09 03:40:36 [config.py:823] This model supports multiple tasks: {'classify', 'generate', 'embed', 'score', 'reward'}. Defaulting to 'generate'.
INFO 07-09 03:40:36 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-09 03:40:36 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.


INFO 07-09 03:40:37 [core.py:455] Waiting for init message from front-end.
INFO 07-09 03:40:37 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, served_model_name=meta-llama/Llama-3.2-1B

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 07-09 03:40:37 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f675c220dc0>
(VllmWorker rank=0 pid=5509) INFO 07-09 03:40:37 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_1172f6ab'), local_subscribe_addr='ipc:///tmp/e60c4d3a-c265-44d2-adc2-80e605960355', remote_subscribe_addr=None, remote_addr_ipv6=False)
WARNING 07-09 03:40:37 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f675c221240>
(VllmWorker rank=1 pid=5510) INFO 07-09 03:40:37 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_4598d5f2'), local_subscribe_addr='ipc:///tmp/b2ca97e0-d8e5-45b7-9373-429

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  🛑 Conversation 4 terminated by user
🔄 Round 4: Processing 2 active conversations


Adding requests:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  🛑 Conversation 3 terminated by user

⏱️  Total time: 7.47 seconds

📊 Results:
   Conv 1: Expected 2 turns, got 2 turns - Quick pasta recipe?...
            Messages in conversation: 6
   Conv 2: Expected 5 turns, got 4 turns - How to make a complex French d...
            Messages in conversation: 10
   Conv 3: Expected 1 turns, got 1 turns - Simple sandwich instructions?...
            Messages in conversation: 4
   Conv 4: Expected 4 turns, got 4 turns - Detailed cake baking process?...
            Messages in conversation: 10
   Conv 5: Expected 3 turns, got 3 turns - Quick salad?...
            Messages in conversation: 8
